# Financial Transaction Fraud Detection & Risk Analytics

EDA → feature engineering → classification → evaluation → risk scoring.

In [ ]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_preprocessing import load_data
from features import engineer_features
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, precision_recall_curve

## 1. Load and inspect data

In [ ]:
df = load_data('../data/raw/creditcard.csv')
print(df.shape)
display(df.head())
print(df['Class'].value_counts())

## 2. Class imbalance

In [ ]:
print(f"Fraud rate: {df.Class.mean()*100:.4f}%")
df.Class.value_counts().sort_index().plot(kind='bar', title='Legitimate vs Fraud')
plt.show()

## 3. Amount analysis

In [ ]:
display(df.groupby('Class')['Amount'].agg(['count','mean','median','max']).round(2))
plt.hist(np.log1p(df.loc[df.Class==0,'Amount']), bins=80, alpha=.6, label='Legitimate')
plt.hist(np.log1p(df.loc[df.Class==1,'Amount']), bins=80, alpha=.6, label='Fraud')
plt.legend(); plt.title('Log Transaction Amount'); plt.show()

## 4. Feature engineering

In [ ]:
X = engineer_features(df.drop(columns=['Class']))
y = df.Class.astype(int)
print(X[['Time','Hour','Amount','AmountLog']].head())

## 5. Train Random Forest

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
model = RandomForestClassifier(n_estimators=100,max_depth=12,min_samples_leaf=2,class_weight='balanced_subsample',random_state=42,n_jobs=-1)
model.fit(X_train,y_train)
prob = model.predict_proba(X_test)[:,1]
pred = (prob>=.50).astype(int)
print(classification_report(y_test,pred,zero_division=0))
print('ROC-AUC:', roc_auc_score(y_test,prob))
print('PR-AUC:', average_precision_score(y_test,prob))

## 6. Confusion matrix and threshold analysis

In [ ]:
print(confusion_matrix(y_test,pred))
precision, recall, thresholds = precision_recall_curve(y_test,prob)
plt.plot(recall,precision); plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('Precision-Recall Curve'); plt.show()

## 7. Save model

In [ ]:
import joblib
joblib.dump(model,'../models/fraud_random_forest.joblib')
print('Saved model.')